# Cluster and Visualize Unlabeled Acoustic Embeddings (PCA + HDBSCAN)

This notebook:
1. Loads the 1536-dim Perch embeddings
2. Reduces dimensions with **PCA** to **256D**
3. Clusters with **HDBSCAN** on PCA embeddings
4. Adds cluster assignments to the manifest
5. Saves PCA embeddings and clustering outputs for review

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from os import environ

from sklearn.decomposition import PCA
import hdbscan

import matplotlib.pyplot as plt
from tqdm import tqdm
import shutil

## 1. Load Embeddings and Manifest

In [2]:
env_dir = environ.get("POSIDONIA_DATASET_DIR")
DATASET_DIR = Path(env_dir) if env_dir else Path("D:/Posidonia Soundscapes/Fondeo 1_Formentera Ille Espardell/Embeddings_2/dataset")

EMBEDDINGS_PATH = DATASET_DIR / "npy_files" / "unlabeled_embeddings.npy"
MANIFEST_PATH = DATASET_DIR / "unlabeled_manifest.csv"

if not EMBEDDINGS_PATH.exists() or not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing files:\n"
        f"  embeddings: {EMBEDDINGS_PATH}\n"
        f"  manifest:   {MANIFEST_PATH}\n"
        f"Set POSIDONIA_DATASET_DIR to the correct dataset folder."
    )

embeddings = np.load(str(EMBEDDINGS_PATH))
manifest_df = pd.read_csv(str(MANIFEST_PATH))

print(f"Loaded {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}")
print(f"Manifest has {len(manifest_df)} rows")
manifest_df.head()

Loaded 392400 embeddings of dimension 1536
Manifest has 392400 rows


,original_audio,embedding_path,segment_path,audio_path,file_name,embedding_dim
0,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-03.wav,1536
1,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-08.wav,1536
2,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-13.wav,1536
3,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-18.wav,1536
4,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-23.wav,1536


## 2. Dimensionality Reduction: PCA

Reduce 1536D embeddings to **256D** PCA components for clustering.

In [3]:
output_dir = Path(EMBEDDINGS_PATH).parent
pca_output_dir = output_dir / "PCA_256D"
pca_output_dir.mkdir(parents=True, exist_ok=True)

pca_file = pca_output_dir / "pca_embeddings_256d.npy"

if pca_file.exists():
    print("Loading pre-computed PCA embeddings...")
    pca_results = np.load(str(pca_file))
    print(f"Loaded PCA embeddings: {pca_results.shape}")
else:
    print("Running PCA to 256D...")
    embeddings_fp32 = embeddings.astype(np.float32, copy=False)
    pca_results = PCA(n_components=256, random_state=42).fit_transform(embeddings_fp32)

    np.save(str(pca_file), pca_results)
    print(f"Saved PCA embeddings to: {pca_file}")

manifest_df['pca_x'] = pca_results[:, 0]
manifest_df['pca_y'] = pca_results[:, 1]
manifest_df['pca_z'] = pca_results[:, 2]

print("PCA embeddings added to manifest.")

Loading pre-computed PCA embeddings...
Loaded PCA embeddings: (392400, 256)
PCA embeddings added to manifest.


## 3. HDBSCAN Clustering on PCA

Cluster the PCA embeddings with HDBSCAN (automatic cluster count + noise handling).

In [ ]:
HDBSCAN_MIN_CLUSTER_SIZE = 80
HDBSCAN_MIN_SAMPLES = 10

pca_labels_file = pca_output_dir / f"pca_hdbscan_labels_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.npy"

if pca_labels_file.exists():
    print("Loading pre-computed PCA HDBSCAN clustering...")
    pca_labels = np.load(str(pca_labels_file))

    if len(pca_labels) != len(pca_results):
        raise ValueError("Loaded labels do not match PCA embedding length.")
else:
    print("Running HDBSCAN clustering on PCA embeddings...")
    clusterer_pca = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom"
    )
    pca_labels = clusterer_pca.fit_predict(pca_results)

    np.save(str(pca_labels_file), pca_labels)
    print(f"Saved clustering results to: {pca_labels_file}")

manifest_df['pca_cluster'] = pca_labels

n_clusters = int(np.sum(np.unique(pca_labels) >= 0))
n_noise = int(np.sum(pca_labels == -1))

print(f"Clustering done! Found {n_clusters} clusters (excluding noise)")
print(f"Noise points: {n_noise}")
print("\nCluster size distribution:")
print(pd.Series(pca_labels).value_counts().sort_index())

manifest_df.head()

Running HDBSCAN clustering on PCA embeddings...


## 4. Sample Strategy: 20 nearest + 20 farthest + 60 random per cluster

For each non-noise PCA cluster, select up to 100 audio samples for review.

In [ ]:
def sample_cluster(cluster_indices, cluster_embeddings, n_nearest=20, n_farthest=20, n_random=60):
    centroid = cluster_embeddings.mean(axis=0)
    dists = np.linalg.norm(cluster_embeddings - centroid, axis=1)

    order = np.argsort(dists)
    nearest_idx = cluster_indices[order[: min(n_nearest, len(order))]]
    farthest_idx = cluster_indices[order[-min(n_farthest, len(order)):]]

    selected = np.unique(np.concatenate([nearest_idx, farthest_idx]))
    remaining_idx = np.setdiff1d(cluster_indices, selected)

    n_random_eff = min(n_random, len(remaining_idx))
    if n_random_eff > 0:
        rng = np.random.default_rng(42)
        random_idx = rng.choice(remaining_idx, size=n_random_eff, replace=False)
    else:
        random_idx = np.array([], dtype=int)

    sampled_idx = np.concatenate([selected, random_idx])
    return np.unique(sampled_idx)

pca_csv = pca_output_dir / f"subsample_pca_hdbscan_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.csv"

if pca_csv.exists():
    print("Loading pre-computed PCA sample indices...")
    pca_subsample_df = pd.read_csv(str(pca_csv))
    pca_subsample_indices = pca_subsample_df['reduced_embeddings_idx'].values
    print(f"Loaded {len(pca_subsample_indices)} PCA samples")
else:
    print("Sampling from PCA clusters...")
    pca_subsample_indices = []

    cluster_ids = [c for c in np.unique(pca_labels) if c >= 0]
    for cluster_id in tqdm(cluster_ids):
        cluster_indices = np.where(pca_labels == cluster_id)[0]
        if len(cluster_indices) == 0:
            continue

        cluster_embeddings = pca_results[cluster_indices]
        sampled = sample_cluster(cluster_indices, cluster_embeddings)
        pca_subsample_indices.extend(sampled.tolist())

    pca_subsample_indices = np.unique(pca_subsample_indices)
    print(f"Selected {len(pca_subsample_indices)} samples from PCA clustering")

    pca_subsample_df = pd.DataFrame({
        'audio_path': manifest_df.iloc[pca_subsample_indices]['audio_path'].values,
        'embedding_path': manifest_df.iloc[pca_subsample_indices]['embedding_path'].values,
        'reduced_embedding_filepath': str(pca_file),
        'reduced_embeddings_idx': pca_subsample_indices,
        'method': 'pca_hdbscan',
        'cluster': pca_labels[pca_subsample_indices],
    })

pca_subsample_df.head()

## 5. Save Results

Save PCA subsamples, labels, and manifest for downstream review.

In [ ]:
pca_output_dir.mkdir(parents=True, exist_ok=True)

pca_csv = pca_output_dir / f"subsample_pca_hdbscan_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.csv"
pca_labels_out = pca_output_dir / f"pca_hdbscan_labels_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.npy"
manifest_out = pca_output_dir / f"manifest_pca_hdbscan_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.csv"

pca_subsample_df.to_csv(pca_csv, index=False)
np.save(pca_labels_out, pca_labels)
manifest_df.to_csv(manifest_out, index=False)

print(f"Saved PCA subsample: {pca_csv}")
print(f"Saved PCA labels: {pca_labels_out}")
print(f"Saved manifest with PCA columns: {manifest_out}")

## 6. Export PCA Audio Samples to Review Folder

Copy sampled audio files to `Review_HDBSCAN/PCA_256D` with cluster folders (`Noise` for -1).

In [ ]:
def convert_wsl_to_windows_path(wsl_path):
    if isinstance(wsl_path, float):
        return wsl_path
    wsl_path = str(wsl_path).replace('\\', '/')
    if wsl_path.startswith('/mnt/'):
        drive = wsl_path[5].upper()
        rest = wsl_path[7:]
        return f"{drive}:\\{rest}".replace('/', '\\')
    return wsl_path

REVIEW_DIR = Path('/mnt/d/Posidonia Soundscapes/Fondeo 1_Formentera Ille Espardell/Embeddings_2/diagnostics/Review_HDBSCAN')
REVIEW_DIR_WINDOWS = Path(convert_wsl_to_windows_path(str(REVIEW_DIR)))

PCA_DEST_WINDOWS = REVIEW_DIR_WINDOWS / 'PCA_256D'
PCA_ALL = PCA_DEST_WINDOWS / 'All'
PCA_CLUSTERS = PCA_DEST_WINDOWS / 'Clusters'

PCA_ALL.mkdir(parents=True, exist_ok=True)
PCA_CLUSTERS.mkdir(parents=True, exist_ok=True)

print(f"PCA All destination (Windows): {PCA_ALL}")
print(f"PCA Clusters destination (Windows): {PCA_CLUSTERS}")

pca_has_files = any(p.is_file() for p in PCA_DEST_WINDOWS.rglob('*'))

if pca_has_files:
    print('\nDetected existing files in PCA destination folder.')
    print('Skipping copy step to avoid duplicating files.')
else:
    pca_csv = pca_output_dir / f"subsample_pca_hdbscan_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.csv"
    print(f"\nReading from: {pca_csv}")

    pca_df = pd.read_csv(str(pca_csv))
    print(f"Loaded {len(pca_df)} PCA samples")

    cluster_ids = sorted(pca_df['cluster'].dropna().astype(int).unique())
    for cluster_id in cluster_ids:
        folder_name = 'Noise' if cluster_id == -1 else str(cluster_id + 1)
        (PCA_CLUSTERS / folder_name).mkdir(exist_ok=True)

    print('\n' + '=' * 60)
    print('Copying PCA audio files...')
    print('=' * 60)
    pca_copied = 0
    pca_failed = 0

    for idx, row in pca_df.iterrows():
        audio_path_windows = convert_wsl_to_windows_path(row['audio_path'])
        src = Path(audio_path_windows)
        cluster_id = int(row['cluster'])
        folder_name = 'Noise' if cluster_id == -1 else str(cluster_id + 1)

        if src.exists():
            dst_all = PCA_ALL / src.name
            dst_cluster = PCA_CLUSTERS / folder_name / src.name

            try:
                shutil.copy2(src, dst_all)
                shutil.copy2(src, dst_cluster)
                pca_copied += 1
            except Exception as e:
                print(f"  Error copying {src.name}: {e}")
                pca_failed += 1
        else:
            print(f"  Source not found: {src}")
            pca_failed += 1

        if (idx + 1) % 100 == 0:
            print(f"  Progress: {idx + 1}/{len(pca_df)}")

    print(f"\nPCA: Copied {pca_copied} files, {pca_failed} failed")

    print('\n' + '=' * 60)
    print('SUMMARY')
    print('=' * 60)
    print(f"PCA All folder ({PCA_ALL}): {len(list(PCA_ALL.glob('*.wav')))} files")

    print('\nPCA Clusters:')
    for cluster_id in cluster_ids:
        folder_name = 'Noise' if cluster_id == -1 else str(cluster_id + 1)
        cluster_folder = PCA_CLUSTERS / folder_name
        file_count = len(list(cluster_folder.glob('*.wav')))
        print(f"  Cluster {folder_name}: {file_count} files")

    print(f"\nTotal copied: {pca_copied} files")
    print(f"Total failed: {pca_failed} files")